# Phase 3 — Hybrid Search + Evaluation

Goal: make retrieval better, then measure how much better. Phase 2 left `hybrid_search` (pgvector + Postgres full-text, fused with RRF) as the retrieval baseline — this phase adds a cross-encoder reranker on top, then (next) RAGAS + LLM-as-judge evaluation comparing baseline vs hybrid vs hybrid+rerank.

This notebook starts with the reranker — the first open item in [docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md)'s Phase 3 backlog.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

## This notebook runs entirely on a local model

Every LLM call below — query rewriting, judging, RAGAS faithfulness/context
precision — runs on `granite4.1:3b` via Ollama, local and free. No API keys,
no rate limits, no per-provider quota math. The tradeoff is CPU latency
(measured where it matters, not guessed) instead of $ or request limits.

The override below is kernel-local — it patches this notebook's in-memory
module state, not `.env`, so whatever's actually running `src.api` (if
anything) keeps its own provider config untouched.

In [2]:
import time
import src.llm
import eval.ragas_eval

for _module in (src.llm, eval.ragas_eval):
    _module.LLM_PROVIDER = "ollama"
    _module.LLM_MODEL = "granite4.1:3b"

print(f"provider={src.llm.LLM_PROVIDER}  model={src.llm.LLM_MODEL}")

provider=ollama  model=granite4.1:3b


## Cross-encoder reranker

`hybrid_search` returns its top-k by RRF-fused rank — a cheap combination of two independent scores (lexical + vector), not a model that actually reads the query and the chunk together. A cross-encoder does: it scores each (query, chunk) pair jointly, which is slower per pair but more accurate at judging relevance. Standard pattern: retrieve a wider top-k cheaply, rerank down to a narrower top-k with the cross-encoder.

`src/rerank.py` adds this: `rerank(query, chunks, top_k=3)`, using `cross-encoder/ms-marco-MiniLM-L-6-v2` (`sentence-transformers`, already a dependency — same library as the embedding model, no new package).

In [3]:
from src.db import get_conn, hybrid_search
from src.embeddings import embed_texts
from src.rerank import rerank

def retrieve(query: str, top_k: int = 5) -> list[dict]:
    embedding = embed_texts([query])[0]
    with get_conn() as conn:
        return hybrid_search(conn, query, embedding, top_k=top_k)

### Case 1 — a straightforward SEC-filing claim

`hybrid_search` top-5, before reranking:

In [4]:
claim = "Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000."

top5 = retrieve(claim, top_k=5)
for r in top5:
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.0328  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0300  [secedgar] Apple Inc. (AAPL) — Total assets
0.0292  [secedgar] Apple Inc. (AAPL) — Revenue


Reranked down to top-3:

In [5]:
top3 = rerank(claim, top5, top_k=3)
for r in top3:
    print(f"{r['rerank_score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

10.2322  [secedgar] Apple Inc. (AAPL) — Revenue
8.2994  [secedgar] Apple Inc. (AAPL) — Revenue
8.2237  [secedgar] Apple Inc. (AAPL) — Revenue


### Case 2 — a fuzzier, definitional claim

Case 1 is easy — the entity/metric keywords in the claim already overlap with the right chunk, so RRF alone likely gets it right. A Wikipedia-sourced definitional claim is a better test: no numeric anchor, more chunks compete on topic similarity alone.

In [6]:
claim2 = (
    "The price-to-earnings ratio is calculated by dividing a company's "
    "market capitalization by its total revenue."
)

top5_2 = retrieve(claim2, top_k=5)
for r in top5_2:
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

0.0328  [wikipedia] Price–earnings ratio
0.0315  [wikipedia] Price–earnings ratio
0.0304  [wikipedia] Price–earnings ratio
0.0289  [wikipedia] Price–earnings ratio
0.0289  [wikipedia] Price–earnings ratio


In [10]:
top3_2 = rerank(claim2, top5_2, top_k=3)
for r in top3_2:
    print(f"{r['rerank_score']:.4f}  [{r['source']}] {r['title']}")

5.3848  [wikipedia] Price–earnings ratio
4.6426  [wikipedia] Price–earnings ratio
1.5628  [wikipedia] Price–earnings ratio


## Measuring it: `eval/compare_retrieval.py`

The two cases above look right by eye, but "looks right" isn't a measurement.
`eval/compare_retrieval.py` scores every retrieval strategy against `data/eval_claims.csv` on two metrics:

- **hit_rate@k** — out of all claims, what fraction had the correct document somewhere in the top-k results? (`hits / total`, e.g. 63/68 = 93%)
- **MRR@k** (Mean Reciprocal Rank) — same claims, but scored by *how high* the correct document ranked, not just whether it showed up. Each claim scores `1/rank` (1st place = 1.0, 2nd = 0.5, 3rd = 0.33...), averaged across all claims. Two strategies can tie on hit_rate but differ on MRR — hit_rate says "found it", MRR says "found it near the top or buried at the bottom."

In [11]:
# import subprocess
# result = subprocess.run(
#     ["uv", "run", "python", "-m", "eval.compare_retrieval"],
#     cwd="..", capture_output=True, text=True,
# )
# print(result.stdout)
# if result.returncode:
#     print("--- stderr ---")
#     print(result.stderr)  # a crash here shows up as empty/truncated stdout otherwise

This reproduces the *original* 5-method table (minsearch/text/vector/
hybrid/hybrid+rerank) documented below. `eval/compare_retrieval.py` now has
2 more methods (`hybrid_rewrite`, `hybrid_rerank_rewrite`, added for the
"Query rewriting" section further down) — running the full script this way
spawns a **separate process**, which doesn't see this notebook's in-memory
override (set at the top) and re-reads `.env` fresh. For the 2 rewrite
methods, use Stage 1 further down instead — it runs in-process, so it
actually uses the local model.

## Eval dataset expansion — the first run was too easy

Eval set grew from 52 to 76 claims (`data/eval_claims.csv`) specifically to
cover `fred` and `wikipedia`, which the original set barely touched —
otherwise hybrid search scored a ceiling 100% hit_rate that hid whatever
reranking contributes.

Baseline on the current 68 non-INSUFFICIENT claims (measured at 6846
chunks — live DB is now 6850, a small known drift):
`hybrid_rrf` hit_rate@5=93% MRR@5=0.873 → `hybrid_rerank` hit_rate@3=93%
MRR@3=0.919.

### Why the misses happen — a source collision, not a vocabulary problem

All 5 remaining misses are FRED claims. Here's what `hybrid_search` actually
retrieves for one of them:

In [12]:
miss_claim = "US GDP in the third quarter of 2019 was about $21.7 trillion."

for r in retrieve(miss_claim, top_k=5):
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

0.0283  [worldbank] GDP (current US$) — Germany
0.0164  [wikipedia] Recession
0.0164  [worldbank] GDP (current US$) — United States
0.0161  [wikipedia] Hedge fund
0.0161  [worldbank] GDP (current US$) — United States


Every result is World Bank's "GDP (current US\$) — United States" or a
generic Wikipedia page — never FRED's `GDP` series chunk. Same underlying
concept, different source, close enough lexically that hybrid search picks
the wrong one — a real problem, not a hypothetical. What to do about it is
the "Query rewriting" section below.

## RAGAS + LLM-as-judge evaluation

Hit_rate/MRR only measure retrieval. They don't check whether the judge's **verdict** actually follows from the evidence it was given, or whether it quietly used its own background knowledge instead. That's what [RAGAS](https://github.com/explodinggradients/ragas) adds — the same LLM-as-judge idea `verifier.py` already uses, but with standardized, pre-built prompts for specific questions:

- **Faithfulness** — does every claim in the judge's answer trace back to the retrieved evidence, or did it add something the evidence doesn't say?
- **Context precision** — was the retrieved evidence actually relevant to the question, or was it noise the judge had to ignore?

Both come from `ragas.metrics.collections` — this is `ragas==0.4.3`, whose public API is a set of typed classes here, not the `from ragas.metrics import faithfulness` style shown in most tutorials (that's an older version's interface).

In [13]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithoutReference, Faithfulness

from eval.ragas_eval import ragas_llm

llm = ragas_llm()
faithfulness = Faithfulness(llm=llm)
context_precision = ContextPrecisionWithoutReference(llm=llm)

Score the Apple revenue case from above — the judge's actual verdict,
against the actual reranked evidence it saw. On local `granite4.1:3b` this
takes on the order of a minute (judge + faithfulness + context precision,
each its own LLM call — see the timing table further down for exact
measurements), not instant:

In [14]:
from src.verifier import verify_claim
from src.claim_extractor import Claim

apple_claim = Claim(text=claim, entity="Apple", metric="Revenue", value=416161000000.0, date="2025-09-27")
verdict = verify_claim(apple_claim)
evidence = [c["content"] for c in top3]
response_text = f"{verdict.verdict}: {verdict.quote or verdict.reasoning}"

f = await faithfulness.ascore(user_input=claim, response=response_text, retrieved_contexts=evidence)
cp = await context_precision.ascore(user_input=claim, response=response_text, retrieved_contexts=evidence)
print("faithfulness:", f.value)
print("context_precision:", cp.value)

faithfulness: 1.0
context_precision: 0.9999999999


### Two prompts, not one

The rubric asks for "LLM evaluation, 2 prompts" — `verifier.py` now has
`VERDICT_PROMPT_V1` (original) and `VERDICT_PROMPT_V2` (asks for step-by-step
`reasoning` *before* deciding — the same pattern the
[LLM Zoomcamp agent-evaluation lesson](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/04-evaluation/lessons/14-agent-evaluation.md)
uses: a `reasoning` field before the score in the judge's own output schema).
`Verdict.reasoning` is now a required field either way, so both prompts can
be compared on the same schema.

`eval/ragas_eval.py` runs both prompts across `eval_claims.csv`, scoring
verdict accuracy (our own metric, vs. `expected_verdict`) plus
faithfulness/context precision (RAGAS) for each:

```bash
uv run python -m eval.ragas_eval
```

Real numbers for both prompts come from Stage 3 further down — it now runs
this comparison locally, in full, instead of the quota-limited partial runs
this section used to document.

## Query rewriting

The FRED miss above is the target: `hybrid_search` keeps returning World
Bank's "GDP (current US\$) — United States" instead of FRED's own `GDP`
series chunk — same concept, different source, close enough lexically that
hybrid search picks the wrong one. `src/query_rewrite.py` adds an LLM step
before retrieval: it rewrites the claim into a search query, naming which of
the four source types (Wikipedia / World Bank / FRED / SEC EDGAR) the claim
most plausibly belongs to when that's ambiguous. `verify_claim()` now calls
it by default (`src/verifier.py`).

Uses the local `granite4.1:3b` override set at the top of this notebook —
nothing provider-specific to configure here.

Evaluated in Stage 1/2 below — the result is mixed, not a clean win (see those sections for numbers).

In [15]:
from src.query_rewrite import rewrite_query

t0 = time.time()
rewritten = rewrite_query(miss_claim)
rewrite_seconds = time.time() - t0
print(f"{rewrite_seconds:.1f}s  ->  {rewritten!r}")

2.8s  ->  'FRED US GDP Q3 2019'


### Sanity check — does the rewrite actually fix the collision?

Same `retrieve()` helper from the reranker section above, plain claim vs rewritten query, side by side:

In [16]:
print("plain claim:")
for r in retrieve(miss_claim, top_k=5):
    print(f"  {r['score']:.4f}  [{r['source']}] {r['title']}")

print("\nrewritten query:")
for r in retrieve(rewritten, top_k=5):
    print(f"  {r['score']:.4f}  [{r['source']}] {r['title']}")

plain claim:
  0.0283  [worldbank] GDP (current US$) — Germany
  0.0164  [wikipedia] Recession
  0.0164  [worldbank] GDP (current US$) — United States
  0.0161  [wikipedia] Hedge fund
  0.0161  [worldbank] GDP (current US$) — United States

rewritten query:
  0.0317  [worldbank] GDP (current US$) — United States
  0.0164  [worldbank] GDP (current US$) — Japan
  0.0164  [fred] Gross Domestic Product (GDP)
  0.0161  [wikipedia] Recession
  0.0161  [fred] Gross Domestic Product (GDP)


The plain claim never surfaces FRED's `GDP` chunk in the top-5 (confirmed in the "Why the misses happen" section above).

**A single run of the cell above is not a reliable check** — re-running it was found to give a *different* rewritten query each time (temperature=0 is requested in `chat_llm()`, but Ollama's decoding for this model doesn't fully pin that down), and the plain-claim retrieval itself isn't perfectly stable either: several World Bank country-GDP chunks ("GDP (current US\$) — Germany/Japan/United States/...") are lexically near-identical, so `ts_rank`/cosine ties between them aren't broken consistently run to run (no tiebreaker `ORDER BY` on `document_chunks.id` in `src/db.py`). One run surfaced FRED at rank 3 with the rewritten query; the very next run didn't surface it at all. **This is exactly why Stage 1 below scores all 68 claims and reports hit_rate/MRR, not a single before/after example** — a one-off anecdote here would be measuring sampling noise, not the rewrite's actual effect.


**Stage 1 has since been run for real** (below): hit_rate came out flat vs. the no-rewrite baseline, MRR came out worse. The single-example variance described above is part of why — but the aggregate result across 68 claims is the one that counts, and it isn't a clean improvement.

### Timing budget — why this notebook stages instead of just running everything

Per-call latency, measured directly on this CPU (not guessed):

| Call | measured |
|---|---|
| `rewrite_query()`, single call | 2.8s (varied 2.6-18s across different runs — see the non-determinism note below) |
| `verify_claim()` full path (rewrite+retrieve+rerank+judge) | ~48s |
| RAGAS `Faithfulness.ascore()` | 50.8s — the most expensive single call |
| RAGAS `ContextPrecisionWithoutReference.ascore()` | 29.6s |

Actual stage runtimes, now that all three have been run at least once:

| Stage | actual |
|---|---|
| 1 — 68 claims × rewrite (cached across both variants) + scoring | 167s + 6s ≈ 3 min — much faster than per-call latency above would suggest; the model stays warm across a batch |
| 2 — 5-claim targeted check | not yet run for real — placeholder `KNOWN_MISS_IDS` matched 0 claims |
| 3 — 76 claims × 2 prompts × (judge+faithfulness+context precision) | done — see Stage 3's results below; total wall-clock wasn't separately timed |

Single-call latency isn't a reliable predictor of batch runtime on this
setup — budget a stage from its own measured run, not from multiplying a
single call's latency.

## Stage 1 — retrieval-only ablation (cheap, no judge)

This is the actual rubric-relevant measurement: "Retrieval evaluation — multiple retrieval approaches are evaluated, and the best one is used" ([project.md](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/project.md#evaluation-criteria)).
Query rewriting is a retrieval-stage change — judging it doesn't need the LLM-as-judge verdict at all, only hit_rate/MRR, same as every other retrieval variant already compared above.

**Pattern borrowed from the course** ([`04-evaluation/code/02-search-eval.ipynb`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/04-evaluation/code/02-search-eval.ipynb)):
`evaluate(ground_truth, search_function)` takes the retrieval strategy as a first-class function argument, so one scoring loop works for every strategy. `eval/compare_retrieval.py` already uses this shape for minsearch/text/vector/hybrid/hybrid+rerank — the cells below just add two more `search_function`s (`hybrid_rewrite`, `hybrid_rerank_rewrite`) rather than reinventing the scoring loop.

Toggle `RUN_STAGE_1` to actually execute — left `False` so re-running this notebook top-to-bottom doesn't cost ~20 minutes by accident.

In [17]:
RUN_STAGE_1 = True

if RUN_STAGE_1:
    from tqdm import tqdm

    from eval.compare_retrieval import is_relevant, load_claims

    claims = load_claims()
    print(f"{len(claims)} claims (INSUFFICIENT rows excluded)")

    rewrite_cache: dict[str, str] = {}

    def cached_rewrite(query: str) -> str:
        if query not in rewrite_cache:
            rewrite_cache[query] = rewrite_query(query)
        return rewrite_cache[query]

    def score(name: str, retrieve_fn, k: int) -> None:
        hits, rr_sum = 0, 0.0
        t0 = time.time()
        for claim in tqdm(claims, desc=name):
            results = retrieve_fn(claim["claim"])[:k]
            for i, row in enumerate(results):
                if is_relevant(row, claim["source_hint"]):
                    hits += 1
                    rr_sum += 1.0 / (i + 1)
                    break
        n = len(claims)
        elapsed = time.time() - t0
        print(f"{name:>21}: hit_rate@{k}={hits}/{n} ({hits / n:.0%})   MRR@{k}={rr_sum / n:.3f}   ({elapsed:.0f}s)")

    score("hybrid_rewrite", lambda q: retrieve(cached_rewrite(q), top_k=5), k=5)
    score("hybrid_rerank_rewrite", lambda q: rerank(cached_rewrite(q), retrieve(cached_rewrite(q), top_k=5), top_k=3), k=3)
else:
    print("RUN_STAGE_1 is False — flip it to run the ~20-minute ablation")

68 claims (INSUFFICIENT rows excluded)


hybrid_rewrite: 100%|██████████| 68/68 [02:47<00:00,  2.46s/it]


       hybrid_rewrite: hit_rate@5=63/68 (93%)   MRR@5=0.765   (167s)


hybrid_rerank_rewrite: 100%|██████████| 68/68 [00:06<00:00, 10.48it/s]

hybrid_rerank_rewrite: hit_rate@3=63/68 (93%)   MRR@3=0.909   (6s)


**Result**: `hybrid_rewrite` hit_rate@5=63/68 (93%) MRR@5=0.765 (167s) →
`hybrid_rerank_rewrite` hit_rate@3=63/68 (93%) MRR@3=0.909 (6s, rewrite
cache reused from the first run).

**Compared against the re-verified current baseline (90%/0.860 → 90%/0.890,
both measured at today's 6850-chunk DB — not the stale 2026-07-11 numbers
this cell originally compared against): hit_rate improves by 2 hits
(90%→93%), and post-rerank MRR improves too (0.890→0.909).** Only the
pre-rerank MRR is worse (0.860→0.765) — `hybrid_rerank_rewrite` is what
`verify_claim()` actually uses, not the un-reranked `hybrid_rewrite`, so the
number that matters in practice moved the right direction. (An earlier
version of this cell compared today's numbers against the old 93%/0.873
baseline and concluded rewriting was flat-to-negative — that was comparing
two different database states, not measuring rewriting itself. See the
non-determinism note above and the re-verification in
`docs/phase-3-evaluation.md` for how the drift was found: 4 extra chunks
between the old and new measurement cost 2 hits on the no-rewrite
baseline.)

### One real check before deciding whether Stage 2 is worth running

Stage 1 shows whether the *right chunk* enters the candidate set — not
whether the reranker ranks it first, or the judge cites it. One real
`verify_claim()` call on `miss_claim` checks that directly:

In [18]:
from src.claim_extractor import Claim

miss_claim_obj = Claim(text=miss_claim, entity="United States", metric="GDP", value=21.7e12, date="2019-Q3")
verdict = verify_claim(miss_claim_obj)
print(f"verdict={verdict.verdict}  source={verdict.source!r}")
print(f"quote={verdict.quote!r}")

verdict=VERIFIED  source='GDP (current US$) — United States'
quote='GDP (current US$) for United States in 2019 was 21539982000000.00.'


## Stage 2 — targeted judge check on the 5 known misses (moderate, optional depth)

The check above confirms the concern directly: rewriting got FRED's chunk
into the candidate set (Stage 1's job), but `verify_claim()` still cited
World Bank — the verdict was correct, the source wasn't. Rewriting fixes
*recall*, not *ranking*. Consistent with Stage 1's aggregate numbers (flat
hit_rate, worse MRR), not a fluke.

**The cell below was already run, but with the `KNOWN_MISS_IDS` placeholder
never filled in — it correctly matched 0 claims and did nothing.** Fill in
the real 5 FRED-miss ids from `data/eval_claims.csv` (cross-reference
against `compare_retrieval.py`'s per-claim misses) before this produces any
signal beyond the one anecdote above.

In [19]:
RUN_STAGE_2 = True

if RUN_STAGE_2:
    import csv
    from pathlib import Path

    from src.claim_extractor import Claim
    from src.verifier import verify_claim

    EVAL_CSV = Path("..") / "data" / "eval_claims.csv"
    with open(EVAL_CSV, newline="", encoding="utf-8") as f:
        all_claims = list(csv.DictReader(f))

    # the 5 FRED claims that missed in "Why the misses happen" above — fill in
    # their ids once `eval/compare_retrieval.py` (or Stage 1) names them explicitly
    KNOWN_MISS_IDS = {"TODO_FILL_IN_AFTER_STAGE_1"}
    known_misses = [row for row in all_claims if row["id"] in KNOWN_MISS_IDS]
    print(f"{len(known_misses)} known-miss claims loaded")

    for row in known_misses:
        claim = Claim(text=row["claim"], entity="", metric="", value=None, date=None)

        without = verify_claim(claim, search_query=claim.text)  # skip rewrite
        with_rw = verify_claim(claim)  # default: rewrite_query() runs internally

        correct_before = without.verdict == row["expected_verdict"]
        correct_after = with_rw.verdict == row["expected_verdict"]
        print(f"[{row['id']}] expected={row['expected_verdict']}")
        print(f"  without rewrite: {without.verdict:<12} source={without.source!r}  {'OK' if correct_before else 'WRONG'}")
        print(f"  with rewrite:    {with_rw.verdict:<12} source={with_rw.source!r}  {'OK' if correct_after else 'WRONG'}")
else:
    print("RUN_STAGE_2 is False — needs KNOWN_MISS_IDS filled in from Stage 1's output first")

0 known-miss claims loaded


## Model comparison — does the rewrite model matter?

Stage 1/2 above used `granite4.1:3b`. Three more local models tried for the
same two stages, to see whether a bigger/different rewrite model actually
fixes what `granite4.1:3b` didn't: `ornith:latest` (9B), `laguna-xs-2.1:latest`
(33B — the same model family used earlier via OpenRouter's free tier),
`granite4.1:8b`. Run as standalone background scripts, not live notebook
cells — each stage takes minutes to hours per model and needs the previous
Ollama model unloaded first (this machine can't hold two of these in RAM at
once), which doesn't fit a linear notebook re-run. The pattern used per
model is in the code cell below; the results table is from those runs, all
against the current 6850-chunk DB (re-verified, not the stale 2026-07-11
baseline used in this notebook's earlier text).

**`ornith:latest` (9B): unusable.** Timed out twice on a single
`rewrite_query()` call — once at the default 180s timeout, once again at
420s after bumping it. Meanwhile `laguna-xs-2.1` (33B, nearly 4x bigger)
answered the same call in 75s. Not a capacity problem — something specific
to this model's decoding or structured-output handling on this setup.
Dropped from the comparison entirely.

**Stage 1 (68 claims, retrieval only):**

| Model | hit_rate@5 | MRR@5 | hit_rate@3 (rerank) | MRR@3 (rerank) |
|---|---|---|---|---|
| *no rewrite (baseline)* | 90% (61/68) | 0.860 | 90% (61/68) | 0.890 |
| `granite4.1:3b` (3.4B) | 93% (63/68) | 0.765 | 93% (63/68) | 0.909 |
| `laguna-xs-2.1` (33B) | 91% (62/68) | 0.756 | 91% (62/68) | 0.890 |
| `granite4.1:8b` (8B) | **96% (65/68)** | 0.777 | **96% (65/68)** | **0.934** |

All three models beat the no-rewrite baseline on hit_rate, and tie-or-beat
it on post-rerank MRR — rewriting genuinely helps retrieval. `granite4.1:8b`
wins by the largest margin. `laguna-xs-2.1` — nearly 10x the parameters of
`granite4.1:3b` — gives the smallest improvement of the three despite being
the biggest model. Size alone doesn't predict rewrite quality; this looks
more like a granite-family-scales-well, laguna-less-so split than a general
"bigger is better" pattern.

**Stage 2 (7 known-miss claims: 53, 54, 55, 56, 57, 58, 62 — verdict
correct / 7):**

| Model | without rewrite | with rewrite | net |
|---|---|---|---|
| `granite4.1:3b` | 2/7 (53, 56) | 2/7 (53, 57) | 0 — churn: gains 57, loses 56 |
| `laguna-xs-2.1` | 1/7 (56) | **3/7** (55, 56, 58) | **+2** |
| `granite4.1:8b` | **3/7** (53, 55, 56) | 2/7 (56, 62) | **-1** — gains 62, loses 53 and 55 |

**The model with the best retrieval numbers (`granite4.1:8b`) has the worst
Stage 2 outcome — rewriting makes its final verdicts *less* accurate, not
more, even though it retrieves the right chunk more often than any other
model.** `laguna-xs-2.1` shows the opposite pattern: worse retrieval (of
the three rewrite models, though still better than no-rewrite), better
downstream verdicts. Retrieval quality (Stage 1) and end-to-end verdict
correctness (Stage 2) aren't the same measurement, and improving one
doesn't reliably improve the other here — plausibly because the same model
that rewrote the query also has to *judge* using whatever it retrieved, and
a different (even objectively better-retrieving) set of evidence chunks can
still trip up that judging step in ways Stage 1's metrics don't see.

**n=7 is small** — one claim flipping changes the percentage by ~14 points,
so none of these Stage 2 deltas should be read as a solid statistical
result. The direction is at least consistent with three independently-run
models landing in three different places (flat, +2, -1), not obviously
noise, but this needs a larger known-miss set to say more than that.

**Bottom line: query rewriting genuinely improves retrieval (Stage 1, all
three models), but that doesn't reliably translate into better final
verdicts (Stage 2, mixed/negative).** Production ships `granite4.1:3b`: it's
the only one of the three that improves retrieval without regressing Stage
2 verdict correctness, and it's the fastest by a wide margin.
`granite4.1:8b` gives the best retrieval numbers but regresses verdicts;
`laguna-xs-2.1` improves verdicts but is far too slow for a
request-response API (~75-100s per rewrite call alone).

In [ ]:
# Pattern used for each model (ornith, laguna-xs-2.1, granite4.1:8b), run as a
# standalone script per model rather than live in this notebook -- see the
# markdown above for why (sequential model loading, minutes-to-hours per run).
#
# import src.llm as llm_module
# from src.config import LLM_PROVIDERS
# LLM_PROVIDERS["ollama"]["timeout"] = 300  # some judge calls exceed the 180s default
# llm_module.LLM_PROVIDER = "ollama"
# llm_module.LLM_MODEL = "<model name>"
#
# Stage 1 reused eval/compare_retrieval.py's is_relevant()/load_claims() with the
# same hybrid_rewrite/hybrid_rerank_rewrite search_functions defined earlier in
# this notebook. Stage 2 reused this notebook's verify_claim(claim, search_query=...)
# pattern against the same 7 claim ids, wrapped in try/except per call (see the
# ollama timeout note above -- a single slow call shouldn't kill the whole run).
#
# Between models: `ollama stop <model>` to force an unload before loading the next
# one -- this machine doesn't have enough RAM to hold two of these at once
# (laguna-xs-2.1 alone is 20GB; loading it alongside anything else pushed swap to
# 100% full and things got unreliable).

## Stage 3 — full RAGAS re-run, both prompts, all claims — done

Ran locally, both prompts, all 76 claims:

| Prompt | Accuracy | Faithfulness* | Context precision |
|---|---|---|---|
| `VERDICT_PROMPT_V1` | **79% (60/76)** | 0.748 | **0.669** |
| `VERDICT_PROMPT_V2` | 74% (56/76) | 0.747 | 0.587 |

*Recomputed after fixing an aggregation bug in `eval/ragas_eval.py`:
RAGAS's `Faithfulness` metric returns `NaN` (not `None`) for a verdict with
no extractable factual statement — mostly `INSUFFICIENT` verdicts — and the
old code summed `NaN` straight into the average, silently turning the
*whole* average into `NaN` instead of just dropping those claims. Fixed to
drop `NaN` the same way it already dropped `None` (3 claims dropped for V1,
10 for V2 — V2 producing more `INSUFFICIENT` verdicts is also most of why
its accuracy is lower).

**V1 wins** — higher accuracy, higher context precision, tied on
faithfulness. `verifier.py`'s current default (`VERDICT_PROMPT_V1`) is
already the right call; no code change needed.

In [ ]:
RUN_STAGE_3 = False  # already run once — see results in the markdown above and this cell's stored output; flip to True only to redo it

if RUN_STAGE_3:
    from eval.ragas_eval import VERDICT_PROMPT_V1, VERDICT_PROMPT_V2, load_claims, run_prompt

    claims = load_claims()  # pass a sample here first if you want to sanity-check timing
    await run_prompt("VERDICT_PROMPT_V1", VERDICT_PROMPT_V1, claims)
    await run_prompt("VERDICT_PROMPT_V2", VERDICT_PROMPT_V2, claims)
else:
    print("RUN_STAGE_3 is False — this is the expensive one, see the markdown above for why it's optional")

--- VERDICT_PROMPT_V1 ---
  [1] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [2] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [3] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [4] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=0.5, context_precision=0.9999999999
  [5] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [6] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=0.6, context_precision=0.9999999999
  [7] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.99999999995
  [8] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [9] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=0.5, context_precision=0.9999999999
  [10] wrong, verdict=INSUFFICIENT (expected VERIFIED),

### Borrowed from the course material, and what wasn't used

Researched three sources before writing this notebook:
[`04-evaluation`](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/04-evaluation),
[`07-project-example`](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/07-project-example),
and [`project.md`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/project.md#evaluation-criteria)'s
rubric text, verbatim.

**Used:**
- `evaluate(ground_truth, search_function)`'s first-class-function shape
  (`04-evaluation` lesson 05) — already how `eval/compare_retrieval.py` is
  built; Stage 1 above just adds two more functions to it.
- Cheap-first / expensive-second staging — not actually present in the
  current `04-evaluation` module (it runs its LLM judge over the *full*
  answer set, no pre-filter), but real in the course's 2025-cohort archive
  (`cohorts/2025/03-evaluation`): cosine-similarity as a free pre-filter,
  LLM-judge on a `df.sample(n=150)` subsample. Adapted here as
  local-model-retrieval-first (Stage 1), judge-second (Stage 2, targeted
  at 5 claims instead of a random sample — we already know exactly which
  claims are in question, a random sample would be strictly worse for this
  specific check).
- Percentage-breakdown result-table style (`07-project-example`'s
  RELEVANT/PARTLY_RELEVANT/NON_RELEVANT table, sampled at 200) — used for
  the hit_rate/MRR-per-strategy tables throughout this notebook rather than
  a single pass/fail number.
- Rubric's own wording on hybrid search — "(at least evaluating it)" —
  applied the same logic to query rewriting: Stage 1 (evaluating it) is
  the rubric requirement, Stage 3 (full re-verification through the judge)
  is not.

**Not used, and why:**
- Cosine-similarity-of-generated-vs-reference-answer — doesn't apply here.
  That technique compares a *generated free-text answer* against a
  reference answer; this project's judge outputs a 3-way classification
  (`VERIFIED`/`REFUTED`/`INSUFFICIENT`) plus a quote, not free text to
  compare by embedding similarity. RAGAS's faithfulness/context-precision
  (already wired in above) is the actual equivalent for this shape of
  output.
- `07-project-example`'s train/val split for boost-parameter tuning — no
  boost parameters in this project's retrieval (RRF fusion has none to
  tune the way minsearch's field-boost dict does), so nothing to split
  data for.

## What's next

- **Resolved**: query rewriting improves retrieval (Stage 1, all 3 models
  beat the re-verified no-rewrite baseline) but doesn't reliably improve
  final verdicts (Stage 2, mixed across models). `granite4.1:3b` ships —
  the only model that improves retrieval without regressing Stage 2, and
  the fastest. See "Model comparison" above.
- **Resolved**: `VERDICT_PROMPT_V1` wins over V2 (79% vs 74% accuracy, 0.669
  vs 0.587 context precision, tied on faithfulness) — `verifier.py`'s
  current default is already correct, no code change needed.
- **Resolved**: the no-rewrite baseline was stale (measured at 6846 chunks,
  live DB is 6850) — re-verified at 90%/0.860 → 90%/0.890, corrected
  throughout this notebook and `docs/phase-3-evaluation.md`.
- Stage 2's 7-claim check is still a small sample (`n=7`) — a larger
  known-miss set would make the Stage 2 model-comparison deltas (0 / +2 /
  -1) more than suggestive.
- Update the README's Phase 3 section (currently still marked "🚧 in
  progress" despite everything above landing) — bullet-list the
  deliverables, name which rubric line each one closes.
- Phase 4: Streamlit UI + Langfuse monitoring

[← Back to README](../README.md)